# V4 训练专用 · 88×88×24 · 条件 CVAE（修复"条件被淹没"）

基于 V3 训练管线。**诊断结论**（见 `V4 实现方案（诊断结论与设计）.md`）：V3 非后验坍缩，
真正病灶是**解码器无视条件**（改 cond 体素变化仅 0.034%，cond/z 影响比 0.025）。

## 相对 V3 的核心变更

| 项目 | V3 | **V4** |
|---|---|---|
| 条件语义 | 房间**占比** | **绝对数量**（软缩放 /4） |
| 条件注入 | 入口 concat 一次 | **CondEncoder 升维(17→64) + FiLM 每层注入** |
| 解码器归一化 | BatchNorm3d | **GroupNorm(affine=False) + FiLM(γ,β)** |
| 编码器 pooling | mean-pool | **mean-pool + 每超类计数** |
| 辅助损失 | 无 | **计数头**（mu→房间数，smooth_l1×0.1） |
| 画布 | 88×88×24 | 88×88×24（不变） |

> 解码器签名仍为 `decoder(z, cond_raw)`（内部做 embed），便于推理包对接。
> KL 保持 V3 设置（坍缩非问题，隔离变量）。

## 步骤
| 步骤 | 作用 |
|---|---|
| Step 0–4 | 依赖、配置、预处理(QC)、V4 建模、训练 |
| Step 1b | 合成拓扑探针 |
| **Step D** | V4-a 诊断（训练后复测：cond/z 影响比应大幅上升） |
| Step 5 | 训练结果摘要 |


In [ ]:

# Step 0: 安装依赖（Colab 首次运行）
!pip -q install torch_geometric


In [ ]:
# Step 1: 全局配置 + 数据工具
import os, json, random, glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.data import HeteroData
from torch_geometric.loader import DataLoader as GraphDataLoader
from torch_geometric.nn import HeteroConv, SAGEConv, GATv2Conv, global_mean_pool

USE_DRIVE = True
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        USE_DRIVE = False

def resolve_data_dir():
    candidates = [
        '/content/drive/MyDrive/master_thesis/data',
        '/content/drive/MyDrive/260509_dataset',
        '/content/data',
        os.path.join(os.getcwd(), 'data'),
        os.path.abspath(os.path.join(os.getcwd(), '..', 'data')),
    ]
    for path in candidates:
        if os.path.isdir(path):
            return path
    return candidates[0]

DATA_DIR = resolve_data_dir()


VOXEL_SIZE = 300.0
RES_X, RES_Y, RES_Z = 88, 88, 24

CHANNEL_MAP = {
    'empty': 0, 'entryway': 1, 'living_room': 2, 'dining_room': 3,
    'kitchen': 4, 'bedroom': 5, 'bathroom': 6, 'corridor': 7,
    'stairs': 8, 'utility': 9, 'balcony': 10, 'multi_purpose': 11,
}
ROOM_TYPES = [k for k in CHANNEL_MAP if k != 'empty']
NUM_CHANNELS = len(CHANNEL_MAP)
NODE_IN_DIM = 8
COND_DIM = 3 + len(ROOM_TYPES) + 3
NUM_ROOM_TYPES = len(ROOM_TYPES)
COND_COUNT_SCALE = 4.0  # V4: 房间数软缩放（4间→1.0，保留 3vs5 差异）
COND_EMBED_DIM = 64     # V4: 条件升维（FiLM 注入）
AUX_COUNT_WEIGHT = 0.1  # V4: 计数辅助损失权重

LIGHTING_ACCESS_MAP = {'direct': 1.0, 'indirect': 0.5, 'none': 0.0}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'设备: {device}')
print(f'数据目录: {DATA_DIR}  存在: {os.path.isdir(DATA_DIR)}')
print(f'节点维={NODE_IN_DIM}  条件维={COND_DIM}')


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_node_supertype(room_type):
    if room_type in ['bedroom', 'living_room', 'dining_room', 'multi_purpose']:
        return 'living'
    if room_type in ['kitchen', 'bathroom', 'utility', 'balcony']:
        return 'service'
    return 'circulation'


def check_hetero_adjacency(r1, r2, tol=150, min_shared_len=300):
    if r1.get('floor', 1) != r2.get('floor', 1):
        if r1['type'] == 'stairs' or r2['type'] == 'stairs':
            ox = max(0, min(r1['box_max'][0], r2['box_max'][0]) - max(r1['box_min'][0], r2['box_min'][0]))
            oy = max(0, min(r1['box_max'][1], r2['box_max'][1]) - max(r1['box_min'][1], r2['box_min'][1]))
            if ox > 0 and oy > 0:
                return 'vertical'
        return None
    if r1.get('floor', 1) == r2.get('floor', 1):
        ox = max(0, min(r1['box_max'][0], r2['box_max'][0]) - max(r1['box_min'][0], r2['box_min'][0]) + tol)
        oy = max(0, min(r1['box_max'][1], r2['box_max'][1]) - max(r1['box_min'][1], r2['box_min'][1]) + tol)
        if (ox > min_shared_len and oy > 0) or (oy > min_shared_len and ox > 0):
            return 'horizontal'
    return None


def effective_lighting_ratio(room):
    dx = room['box_max'][0] - room['box_min'][0]
    dy = room['box_max'][1] - room['box_min'][1]
    floor_area = max(dx * dy, 1.0)
    eff = room.get('effective_lighting', [])
    total = sum(float(e.get('area', 0)) * float(e.get('attenuation', 1.0)) for e in eff)
    return min(total / floor_area, 5.0) / 5.0


def build_room_features(room, build_min, b_size):
    dx = room['box_max'][0] - room['box_min'][0]
    dy = room['box_max'][1] - room['box_min'][1]
    area = (dx * dy) / 1e6
    aspect = max(dx, dy) / max(min(dx, dy), 1.0)
    cx = ((room['box_min'][0] + room['box_max'][0]) / 2 - build_min[0]) / (b_size[0] + 1e-5)
    cy = ((room['box_min'][1] + room['box_max'][1]) / 2 - build_min[1]) / (b_size[1] + 1e-5)
    cz = ((room['box_min'][2] + room['box_max'][2]) / 2 - build_min[2]) / (b_size[2] + 1e-5)
    lighting_access = LIGHTING_ACCESS_MAP.get(room.get('lighting_access', 'none'), 0.0)
    lighting_priority = float(room.get('lighting_priority', 0)) / 10.0
    lighting_ratio = effective_lighting_ratio(room)
    return [area, aspect, cx, cy, cz, lighting_access, lighting_priority, lighting_ratio]


def build_condition_vector(data):
    meta = data.get('metadata', {})
    stats = meta.get('stats', {})
    bsize = meta.get('building_size', {'x': 1.0, 'y': 1.0, 'z': 1.0})
    rooms = data.get('rooms', [])
    total = max(len(rooms), 1)

    cond = [
        float(bsize.get('x', 1.0)) / 30000.0,
        float(bsize.get('y', 1.0)) / 30000.0,
        float(bsize.get('z', 1.0)) / 9000.0,
    ]
    for rt in ROOM_TYPES:
        cond.append(float(stats.get(rt, 0)) / COND_COUNT_SCALE)  # V4: 绝对数量

    direct = indirect = none = 0
    for r in rooms:
        acc = r.get('lighting_access', 'none')
        if acc == 'direct':
            direct += 1
        elif acc == 'indirect':
            indirect += 1
        else:
            none += 1
    cond.extend([direct / COND_COUNT_SCALE, indirect / COND_COUNT_SCALE, none / COND_COUNT_SCALE])  # V4
    return cond


def json_to_sample(data):
    rooms = data.get('rooms', [])
    if not rooms:
        return None

    all_coords = np.array([r['box_min'] for r in rooms] + [r['box_max'] for r in rooms])
    build_min = all_coords.min(axis=0)
    build_max = all_coords.max(axis=0)
    b_size = build_max - build_min

    hetero = HeteroData()
    nodes_dict = {'living': [], 'service': [], 'circulation': []}
    id_to_idx = {}

    for r in rooms:
        ntype = get_node_supertype(r['type'])
        id_to_idx[r['id']] = (ntype, len(nodes_dict[ntype]))
        nodes_dict[ntype].append(build_room_features(r, build_min, b_size))

    for ntype, feats in nodes_dict.items():
        hetero[ntype].x = torch.tensor(feats, dtype=torch.float32) if feats else torch.empty((0, NODE_IN_DIM))

    edges_dict = {}
    for i in range(len(rooms)):
        for j in range(i + 1, len(rooms)):
            r1, r2 = rooms[i], rooms[j]
            rel = check_hetero_adjacency(r1, r2)
            if not rel:
                continue
            t1, idx1 = id_to_idx[r1['id']]
            t2, idx2 = id_to_idx[r2['id']]
            for src_t, dst_t, si, di in ((t1, t2, idx1, idx2), (t2, t1, idx2, idx1)):
                key = (src_t, rel, dst_t)
                edges_dict.setdefault(key, []).append([si, di])

    for key, elist in edges_dict.items():
        hetero[key].edge_index = torch.tensor(elist, dtype=torch.long).t().contiguous()

    grid = np.zeros((RES_X, RES_Y, RES_Z), dtype=np.int8)
    phys_center_xy = (build_min[:2] + build_max[:2]) / 2.0
    offset_xy = np.array([RES_X * VOXEL_SIZE / 2, RES_Y * VOXEL_SIZE / 2]) - phys_center_xy
    z_min_phys = build_min[2]

    for r in rooms:
        ch = CHANNEL_MAP.get(r['type'], 0)
        ix_min = int((r['box_min'][0] + offset_xy[0]) / VOXEL_SIZE)
        ix_max = int((r['box_max'][0] + offset_xy[0]) / VOXEL_SIZE)
        iy_min = int((r['box_min'][1] + offset_xy[1]) / VOXEL_SIZE)
        iy_max = int((r['box_max'][1] + offset_xy[1]) / VOXEL_SIZE)
        iz_min = int((r['box_min'][2] - z_min_phys) / VOXEL_SIZE)
        iz_max = int((r['box_max'][2] - z_min_phys) / VOXEL_SIZE)
        grid[max(0, ix_min):min(RES_X, ix_max), max(0, iy_min):min(RES_Y, iy_max), max(0, iz_min):min(RES_Z, iz_max)] = ch

    one_hot = np.zeros((NUM_CHANNELS, RES_X, RES_Y, RES_Z), dtype=np.float32)
    for c in range(NUM_CHANNELS):
        one_hot[c] = (grid == c).astype(np.float32)

    cond = torch.tensor(build_condition_vector(data), dtype=torch.float32)
    return {'graph': hetero, 'voxel': torch.tensor(one_hot, dtype=torch.float32), 'condition': cond}


def dice_loss_with_logits(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    dims = (0, 2, 3, 4)
    inter = (probs * targets).sum(dims)
    union = probs.sum(dims) + targets.sum(dims)
    dice = (2 * inter + eps) / (union + eps)
    return 1.0 - dice.mean()


def graph_batch_size(batch):
    return int(getattr(batch, 'num_graphs', 1))


def graph_batch_dict(batch):
    if hasattr(batch, 'batch_dict'):
        return batch.batch_dict
    bd = {}
    for ntype in batch.node_types:
        if batch[ntype].x.size(0) > 0:
            if hasattr(batch[ntype], 'batch'):
                bd[ntype] = batch[ntype].batch
            else:
                bd[ntype] = torch.zeros(
                    batch[ntype].x.size(0), dtype=torch.long, device=batch[ntype].x.device
                )
    return bd


def graph_condition(batch):
    bs = graph_batch_size(batch)
    if not hasattr(batch, 'condition'):
        raise AttributeError('batch 缺少 condition：请用 prepare_graph_batch(hg, condition=tensor)')
    cond = batch.condition
    if cond.dim() == 1:
        cond = cond.unsqueeze(0)
    return cond.view(bs, -1)


def prepare_graph_batch(hg, condition=None):
    """将 condition 挂到 HeteroData 并确保 Batch 后仍可读取。"""
    if condition is not None:
        cond = condition.unsqueeze(0) if condition.dim() == 1 else condition
        hg.condition = cond
    batch = next(iter(GraphDataLoader([hg], batch_size=1)))
    if condition is not None and not hasattr(batch, 'condition'):
        batch.condition = hg.condition
    return batch


def forward_model(model, batch, condition=None):
    """显式传入 condition，避免 HeteroDataBatch 丢字段。"""
    cond = condition if condition is not None else graph_condition(batch)
    if cond.dim() == 1:
        cond = cond.unsqueeze(0)
    mu, logvar = model.encoder(
        batch.x_dict, batch.edge_index_dict, graph_batch_dict(batch)
    )
    z = model.reparameterize(mu, logvar)
    logits = model.decoder(z, cond.to(z.device))
    return logits, mu, logvar

from datetime import datetime

TOTAL_JSON_COUNT = 0  # 在下方 v1 配置块中重新统计
TRAIN_SAMPLE_COUNT = None
TRAIN_TIMESTAMP = None
EXPERIMENT_META = {}


def set_experiment_meta(train_count, total_count=None, timestamp=None, extra=None):
    global TRAIN_SAMPLE_COUNT, TRAIN_TIMESTAMP, EXPERIMENT_META, TOTAL_JSON_COUNT
    TRAIN_SAMPLE_COUNT = int(train_count)
    if total_count is not None:
        TOTAL_JSON_COUNT = int(total_count)
    TRAIN_TIMESTAMP = timestamp or datetime.now().strftime('%Y%m%d_%H%M%S')
    EXPERIMENT_META = {
        'train_samples': TRAIN_SAMPLE_COUNT,
        'total_json': TOTAL_JSON_COUNT,
        'trained_at': TRAIN_TIMESTAMP,
    }
    if extra:
        EXPERIMENT_META.update(extra)
    return EXPERIMENT_META


def experiment_report_name(prefix, ext='csv'):
    ts = TRAIN_TIMESTAMP or datetime.now().strftime('%Y%m%d_%H%M%S')
    n_train = TRAIN_SAMPLE_COUNT if TRAIN_SAMPLE_COUNT is not None else 'NA'
    n_total = TOTAL_JSON_COUNT if TOTAL_JSON_COUNT is not None else 'NA'
    return f"{prefix}_json{n_total}_train{n_train}_{ts}.{ext}"


def experiment_report_path(prefix, subdir='eval_reports', ext='csv'):
    out = os.path.join(DATA_DIR, subdir)
    os.makedirs(out, exist_ok=True)
    return os.path.join(out, experiment_report_name(prefix, ext))


# ========== V3 训练配置（88×88×24）==========
JSON_ROOT = os.path.join(DATA_DIR, 'processed')
OUT_DIR = os.path.join(DATA_DIR, 'processed_tensors_v4')
# Drive 空间不足时可改 Colab 本地（会话结束丢失）：
# OUT_DIR = '/content/processed_tensors_v4'

TENSOR_CACHE_VERSION = 'v4_tensors_88x88x24'
TRAIN_CACHE_VERSION = 'v4_train_88x88x24'
CACHE_VERSION = TRAIN_CACHE_VERSION

WEIGHT_FILENAME = 'spatial_modal_cvae_v4_88x88x24.pth'
WEIGHT_PROBE_FILENAME = 'spatial_modal_cvae_v4_88x88x24_probe_best.pth'
CHECKPOINT_FILENAME = 'spatial_modal_cvae_v4_88x88x24_checkpoint.pt'
SPLIT_FILENAME = 'train_val_split_v4.json'

RES_X, RES_Y, RES_Z = 88, 88, 24
VOXEL_SIZE = 300.0
DECODER_INIT_X, DECODER_INIT_Y, DECODER_INIT_Z = 6, 6, 2

BATCH_SIZE = 4
ACCUM_STEPS = 4
USE_AMP = True
EPOCHS = 90
PATIENCE = 18
LOG_EVERY = 5
LR = 0.001
WEIGHT_DECAY = 0.0001
DICE_WEIGHT = 0.3
KL_WEIGHT_MAX = 0.005
KL_ANNEAL_EPOCHS = 40
WARMUP_EPOCHS = 5
WARMUP_LR_START = 1e-05

EDGE_DROP_PROB = 0.2
EDGE_ADD_PROB = 0.05
NODE_JITTER = 0.02
AUGMENT_PROB = 0.8
PROBE_EVERY = 10

RESUME_IF_CHECKPOINT = True
FORCE_NEW_TRAINING = False
MANUAL_RESUME_EPOCH = 0

VAL_RATIO = 0.2
SPLIT_SEED = 42

RUNS_LOG_PATH = os.path.join(DATA_DIR, 'eval_reports', 'training_runs.jsonl')


def discover_json_files(root=None):
    root = root or JSON_ROOT
    if not os.path.isdir(root):
        return []
    return sorted(
        f for f in glob.glob(os.path.join(root, '**', 'house_*.json'), recursive=True)
        if not f.endswith('_topology.json')
    )


def split_path():
    return os.path.join(DATA_DIR, SPLIT_FILENAME)


def weight_path():
    return os.path.join(DATA_DIR, WEIGHT_FILENAME)


def probe_weight_path():
    return os.path.join(DATA_DIR, WEIGHT_PROBE_FILENAME)


def checkpoint_path():
    return os.path.join(DATA_DIR, CHECKPOINT_FILENAME)


def load_train_val_split(pt_names, seed=SPLIT_SEED, val_ratio=VAL_RATIO):
    path = split_path()
    if os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as f:
            saved = json.load(f)
        train_names = saved.get('train', [])
        val_names = saved.get('val', [])
        known = set(pt_names)
        train_names = [n for n in train_names if n in known]
        val_names = [n for n in val_names if n in known]
        if train_names or val_names:
            return train_names, val_names, saved
    rng = random.Random(seed)
    names = list(pt_names)
    rng.shuffle(names)
    n_val = max(1, int(len(names) * val_ratio))
    val_names = names[-n_val:]
    train_names = names[:-n_val]
    meta = {
        'seed': seed,
        'val_ratio': val_ratio,
        'total': len(names),
        'train_count': len(train_names),
        'val_count': len(val_names),
        'cache_version': CACHE_VERSION,
        'grid': '88x88x24',
    }
    with open(path, 'w', encoding='utf-8') as f:
        json.dump({'train': train_names, 'val': val_names, 'meta': meta}, f, indent=2, ensure_ascii=False)
    return train_names, val_names, meta


def resolve_source_json(sample, pt_basename=None):
    if isinstance(sample, dict) and sample.get('source_json') and os.path.exists(sample['source_json']):
        return sample['source_json']
    if pt_basename:
        cand = os.path.join(DATA_DIR, pt_basename.replace('.pt', '.json'))
        if os.path.exists(cand):
            return cand
    return None


def augment_batch_topology(batch, edge_drop_prob=EDGE_DROP_PROB, node_jitter=NODE_JITTER, edge_add_prob=EDGE_ADD_PROB):
    """拓扑加噪：边 dropout + 同图假边注入 + 节点坐标微扰。"""
    for edge_key in batch.edge_types:
        store = batch[edge_key]
        ei = store.edge_index
        if ei.numel() == 0:
            continue
        keep = torch.rand(ei.size(1), device=ei.device) > edge_drop_prob
        if not keep.any():
            keep = torch.zeros(ei.size(1), dtype=torch.bool, device=ei.device)
            keep[0] = True
        store.edge_index = ei[:, keep]
        if edge_add_prob > 0:
            src_type, dst_type = edge_key[0], edge_key[2]
            if (hasattr(batch[src_type], 'x') and batch[src_type].x is not None and batch[src_type].x.numel() > 0
                and hasattr(batch[dst_type], 'x') and batch[dst_type].x is not None and batch[dst_type].x.numel() > 0):
                n_src = batch[src_type].x.size(0)
                n_dst = batch[dst_type].x.size(0)
                src_graph = batch[src_type].batch if hasattr(batch[src_type], 'batch') else torch.zeros(n_src, dtype=torch.long, device=ei.device)
                dst_graph = batch[dst_type].batch if hasattr(batch[dst_type], 'batch') else torch.zeros(n_dst, dtype=torch.long, device=ei.device)
                existing = set()
                for col in range(store.edge_index.size(1)):
                    existing.add((int(store.edge_index[0, col]), int(store.edge_index[1, col])))
                fake = []
                n_add = max(1, int(store.edge_index.size(1) * edge_add_prob))
                attempts = 0
                while len(fake) < n_add and attempts < n_add * 15:
                    si = torch.randint(0, n_src, (1,)).item()
                    di = torch.randint(0, n_dst, (1,)).item()
                    if src_graph[si] == dst_graph[di] and (si, di) not in existing:
                        fake.append([si, di])
                        existing.add((si, di))
                    attempts += 1
                if fake:
                    fake_tensor = torch.tensor(fake, dtype=torch.long, device=ei.device).t().contiguous()
                    store.edge_index = torch.cat([store.edge_index, fake_tensor], dim=1)
    for ntype in batch.node_types:
        x = batch[ntype].x
        if x is None or x.numel() == 0:
            continue
        noise = torch.zeros_like(x)
        if x.size(1) >= 5:
            noise[:, 2:5] = torch.randn(x.size(0), 3, device=x.device) * node_jitter
        batch[ntype].x = (x + noise).clamp(0.0, 5.0)
    return batch


def qc_sample_voxelization(data):
    """统计体素化裁切与占用率，用于 Step 2 数据集 QC。"""
    rooms = data.get('rooms', [])
    if not rooms:
        return None
    all_coords = np.array([r['box_min'] for r in rooms] + [r['box_max'] for r in rooms])
    build_min = all_coords.min(axis=0)
    build_max = all_coords.max(axis=0)
    bsize = build_max - build_min
    phys_center_xy = (build_min[:2] + build_max[:2]) / 2.0
    offset_xy = np.array([RES_X * VOXEL_SIZE / 2, RES_Y * VOXEL_SIZE / 2]) - phys_center_xy
    z_min_phys = build_min[2]
    clipped_rooms = 0
    for r in rooms:
        ix_min = int((r['box_min'][0] + offset_xy[0]) / VOXEL_SIZE)
        ix_max = int((r['box_max'][0] + offset_xy[0]) / VOXEL_SIZE)
        iy_min = int((r['box_min'][1] + offset_xy[1]) / VOXEL_SIZE)
        iy_max = int((r['box_max'][1] + offset_xy[1]) / VOXEL_SIZE)
        iz_min = int((r['box_min'][2] - z_min_phys) / VOXEL_SIZE)
        iz_max = int((r['box_max'][2] - z_min_phys) / VOXEL_SIZE)
        raw_ix_min = ix_min
        raw_iy_min = iy_min
        raw_iz_min = iz_min
        raw_ix_max = ix_max
        raw_iy_max = iy_max
        raw_iz_max = iz_max
        ix_min = max(0, ix_min)
        iy_min = max(0, iy_min)
        iz_min = max(0, iz_min)
        ix_max = min(RES_X, ix_max)
        iy_max = min(RES_Y, iy_max)
        iz_max = min(RES_Z, iz_max)
        if (raw_ix_min < 0 or raw_iy_min < 0 or raw_iz_min < 0
                or raw_ix_max > RES_X or raw_iy_max > RES_Y or raw_iz_max > RES_Z
                or ix_max <= ix_min or iy_max <= iy_min or iz_max <= iz_min):
            clipped_rooms += 1
    sample = json_to_sample(data)
    if sample is None:
        return None
    occ = int((sample['voxel'].sum(dim=0) > 0).sum().item())
    total_vox = RES_X * RES_Y * RES_Z
    return {
        'building_x': float(bsize[0]),
        'building_y': float(bsize[1]),
        'building_z': float(bsize[2]),
        'clipped_rooms': clipped_rooms,
        'total_rooms': len(rooms),
        'n_occ': occ,
        'occ_ratio': occ / total_vox,
    }


def append_training_run_log(record):
    os.makedirs(os.path.dirname(RUNS_LOG_PATH), exist_ok=True)
    with open(RUNS_LOG_PATH, 'a', encoding='utf-8') as f:
        f.write(json.dumps(record, ensure_ascii=False) + chr(10))


TOTAL_JSON_COUNT = len(discover_json_files())
print(f'JSON 根目录: {JSON_ROOT}  发现 JSON: {TOTAL_JSON_COUNT}')
print(f'缓存目录: {OUT_DIR}  权重: {WEIGHT_FILENAME}')
print(f'画布: {RES_X}×{RES_Y}×{RES_Z}  体素: {VOXEL_SIZE}mm  有效batch≈{BATCH_SIZE * ACCUM_STEPS}')


In [ ]:
# Step 1b: 合成拓扑工具（训练探针（监控条件生成能力））
import math
import networkx as nx

DEFAULT_ROOM_SIZE = {
    'entryway': (2400, 2400, 3000),
    'living_room': (6000, 4500, 3000),
    'dining_room': (3600, 3300, 3000),
    'kitchen': (3300, 3000, 3000),
    'bedroom': (3600, 3600, 3000),
    'bathroom': (2400, 2400, 3000),
    'corridor': (1800, 2400, 3000),
    'stairs': (3000, 3000, 6000),
    'utility': (2400, 2400, 3000),
    'balcony': (3000, 1800, 3000),
    'multi_purpose': (3600, 3300, 3000),
}

SYNTHETIC_PROBE_SPECS = [
    {'name': 'compact_2f', 'site_x': 15000, 'site_y': 12000, 'seed': 11,
     'room_counts': {'entryway': 1, 'living_room': 1, 'dining_room': 1, 'kitchen': 1,
                     'bedroom': 2, 'bathroom': 1, 'corridor': 1, 'stairs': 1}},
    {'name': 'standard_3br', 'site_x': 18000, 'site_y': 15000, 'seed': 22,
     'room_counts': {'entryway': 1, 'living_room': 1, 'dining_room': 1, 'kitchen': 1,
                     'bedroom': 3, 'bathroom': 2, 'corridor': 2, 'stairs': 1, 'balcony': 1}},
    {'name': 'large_4br', 'site_x': 21000, 'site_y': 18000, 'seed': 33,
     'room_counts': {'entryway': 1, 'living_room': 1, 'dining_room': 1, 'kitchen': 1,
                     'bedroom': 4, 'bathroom': 2, 'corridor': 2, 'stairs': 1, 'balcony': 1, 'utility': 1}},
]


def snap_modulus(v):
    return round(float(v) / VOXEL_SIZE) * VOXEL_SIZE


def build_user_request(site_x, site_y, room_counts, site_z=6000):
    counts = {k: int(v) for k, v in room_counts.items() if int(v) > 0}
    return {
        'site_x': float(site_x), 'site_y': float(site_y), 'site_z': float(site_z),
        'room_counts': counts,
    }


def build_program_topology(room_counts, seed=42):
    G = nx.Graph()
    nodes = []
    bath_i = corr_i = 0
    for r_type, count in room_counts.items():
        for i in range(count):
            nid = f"{r_type}_{i}"
            if r_type in ['entryway', 'living_room', 'dining_room', 'kitchen']:
                floor = 1
            elif r_type in ['bedroom', 'balcony']:
                floor = 2
            elif r_type == 'bathroom':
                floor = 1 if bath_i % 2 == 0 else 2
                bath_i += 1
            elif r_type == 'corridor':
                floor = 1 if corr_i % 2 == 0 else 2
                corr_i += 1
            elif r_type == 'stairs':
                floor = '1&2'
            else:
                floor = 1
            nodes.append((nid, r_type, floor))
            G.add_node(nid, type=r_type, floor=floor)

    f1_corr = [n for n, t, f in nodes if t == 'corridor' and f == 1]
    f2_corr = [n for n, t, f in nodes if t == 'corridor' and f == 2]
    f1_hub = f1_corr[0] if f1_corr else next((n for n, t, f in nodes if t == 'living_room'), nodes[0][0])
    f2_hub = f2_corr[0] if f2_corr else next((n for n, t, f in nodes if t == 'bedroom'), nodes[-1][0])

    edge_types = {}
    for nid, r_type, floor in nodes:
        if r_type == 'stairs':
            continue
        hub = f1_hub if floor == 1 else f2_hub
        if nid != hub:
            G.add_edge(nid, hub)
            edge_types[(nid, hub)] = 'horizontal'
            edge_types[(hub, nid)] = 'horizontal'

    stairs = [n for n, t, f in nodes if t == 'stairs']
    if stairs:
        st = stairs[0]
        if f1_hub != st:
            G.add_edge(st, f1_hub)
            edge_types[(st, f1_hub)] = 'vertical'
        if f2_hub != st and f2_hub != f1_hub:
            G.add_edge(st, f2_hub)
            edge_types[(st, f2_hub)] = 'vertical'

    pos = nx.spring_layout(G, seed=seed, k=1.2)
    return G, pos, nodes, edge_types


def layout_rooms_from_program(user_req, seed=42):
    random.seed(seed)
    np.random.seed(seed)
    G, pos, nodes, edge_types = build_program_topology(user_req['room_counts'], seed=seed)
    sx, sy = user_req['site_x'], user_req['site_y']
    rooms = []
    for nid, r_type, floor in nodes:
        w, d, h = DEFAULT_ROOM_SIZE.get(r_type, (3600, 3600, 3000))
        px, py = pos[nid]
        cx = (px + 1) / 2 * (sx * 0.7) + sx * 0.15
        cy = (py + 1) / 2 * (sy * 0.7) + sy * 0.15
        cx, cy = snap_modulus(cx), snap_modulus(cy)
        w, d = snap_modulus(w), snap_modulus(d)
        z0 = 0 if floor == 1 or floor == '1&2' else 3000
        z1 = z0 + h
        rooms.append({
            'id': nid, 'type': r_type, 'floor': 1 if floor == '1&2' else floor,
            'box_min': [max(0, cx - w / 2), max(0, cy - d / 2), z0],
            'box_max': [min(sx, cx + w / 2), min(sy, cy + d / 2), z1],
            'lighting_access': 'direct' if r_type in ['living_room', 'bedroom', 'balcony'] else 'indirect',
            'lighting_priority': 8 if r_type in ['living_room', 'bedroom'] else 4,
            'effective_lighting': [],
        })
    return rooms, G, pos, edge_types


def request_to_house_json(user_req, rooms):
    stats = {t: 0 for t in ROOM_TYPES}
    for r in rooms:
        stats[r['type']] = stats.get(r['type'], 0) + 1
    return {
        'metadata': {
            'stats': stats,
            'total_rooms': len(rooms),
            'building_size': {'x': user_req['site_x'], 'y': user_req['site_y'], 'z': user_req['site_z']},
            'constraints': {'modulus': int(VOXEL_SIZE)},
        },
        'rooms': rooms,
    }


@torch.no_grad()
def synthetic_graph_forward(user_req, model, seed=42):
    rooms, _, _, _ = layout_rooms_from_program(user_req, seed=seed)
    data = request_to_house_json(user_req, rooms)
    sample = json_to_sample(data)
    if sample is None:
        return 0, 0.0
    hg = sample['graph']
    batch = prepare_graph_batch(hg, condition=sample['condition']).to(device)
    model.eval()
    logits, _, _ = forward_model(model, batch, sample['condition'])
    pred = torch.argmax(logits[0], dim=0).cpu().numpy()
    n_occ = int((pred > 0).sum())
    occ_ratio = float(n_occ / max(pred.size, 1))
    return n_occ, occ_ratio


@torch.no_grad()
def run_synthetic_probe_eval(model, specs=None, verbose=True):
    specs = specs or SYNTHETIC_PROBE_SPECS
    results = []
    for spec in specs:
        req = build_user_request(spec['site_x'], spec['site_y'], spec['room_counts'])
        n_occ, occ_ratio = synthetic_graph_forward(req, model, seed=spec.get('seed', 42))
        row = {'name': spec['name'], 'n_occ': n_occ, 'occ_ratio': occ_ratio}
        results.append(row)
        if verbose:
            print(f"  [{spec['name']}] 非空体素={n_occ} ({occ_ratio*100:.4f}%)")
    mean_occ = float(np.mean([r['n_occ'] for r in results]))
    hit = sum(1 for r in results if r['n_occ'] > 0)
    return {'per_probe': results, 'mean_n_occ': mean_occ, 'nonempty_hits': hit, 'total': len(results)}


@torch.no_grad()
def generate_voxels_from_request(user_req, model, seed=42):
    rooms, G, pos, edge_types = layout_rooms_from_program(user_req, seed=seed)
    data = request_to_house_json(user_req, rooms)
    sample = json_to_sample(data)
    hg = sample['graph']
    batch = prepare_graph_batch(hg, condition=sample['condition']).to(device)
    model.eval()
    logits, _, _ = forward_model(model, batch, sample['condition'])
    pred_cls = torch.argmax(logits[0], dim=0).cpu().numpy()
    return pred_cls, sample, rooms, G, pos


def count_components_3d(mask):
    if not mask.any():
        return 0
    visited = np.zeros(mask.shape, dtype=bool)
    n_comp = 0
    xs, ys, zs = np.where(mask)
    for x, y, z in zip(xs, ys, zs):
        if visited[x, y, z]:
            continue
        n_comp += 1
        stack = [(int(x), int(y), int(z))]
        visited[x, y, z] = True
        while stack:
            cx, cy, cz = stack.pop()
            for dx, dy, dz in ((1,0,0),(-1,0,0),(0,1,0),(0,-1,0),(0,0,1),(0,0,-1)):
                nx, ny, nz = cx+dx, cy+dy, cz+dz
                if 0 <= nx < mask.shape[0] and 0 <= ny < mask.shape[1] and 0 <= nz < mask.shape[2]:
                    if mask[nx, ny, nz] and not visited[nx, ny, nz]:
                        visited[nx, ny, nz] = True
                        stack.append((nx, ny, nz))
    return n_comp


def voxel_grid_phys_offset(rooms):
    """与 json_to_sample 编码一致的 XY/Z 偏移，用于体素→物理坐标反变换。"""
    if not rooms:
        return np.array([0.0, 0.0]), 0.0
    all_coords = np.array([r['box_min'] for r in rooms] + [r['box_max'] for r in rooms])
    build_min = all_coords.min(axis=0)
    build_max = all_coords.max(axis=0)
    phys_center_xy = (build_min[:2] + build_max[:2]) / 2.0
    offset_xy = np.array([RES_X * VOXEL_SIZE / 2, RES_Y * VOXEL_SIZE / 2]) - phys_center_xy
    return offset_xy, float(build_min[2])


def voxels_to_boxes(pred_cls, user_req=None, rooms=None):
    TYPE_NAMES_LOCAL = {v: k for k, v in CHANNEL_MAP.items() if v > 0}
    offset_xy, z_min_phys = voxel_grid_phys_offset(rooms) if rooms else (np.array([0.0, 0.0]), 0.0)
    boxes = []
    for cid in range(1, NUM_CHANNELS):
        mask = pred_cls == cid
        if not mask.any():
            continue
        visited = np.zeros(mask.shape, dtype=bool)
        xs, ys, zs = np.where(mask)
        for x, y, z in zip(xs, ys, zs):
            if visited[x, y, z]:
                continue
            stack = [(int(x), int(y), int(z))]
            comp = []
            visited[x, y, z] = True
            while stack:
                cx, cy, cz = stack.pop()
                comp.append((cx, cy, cz))
                for dx, dy, dz in ((1,0,0),(-1,0,0),(0,1,0),(0,-1,0),(0,0,1),(0,0,-1)):
                    nx, ny, nz = cx+dx, cy+dy, cz+dz
                    if 0 <= nx < mask.shape[0] and 0 <= ny < mask.shape[1] and 0 <= nz < mask.shape[2]:
                        if mask[nx, ny, nz] and not visited[nx, ny, nz]:
                            visited[nx, ny, nz] = True
                            stack.append((nx, ny, nz))
            arr = np.array(comp)
            ix0, iy0, iz0 = arr.min(axis=0)
            ix1, iy1, iz1 = arr.max(axis=0) + 1
            rtype = TYPE_NAMES_LOCAL[cid]
            boxes.append({
                'type': rtype,
                'box_min': [
                    snap_modulus(ix0 * VOXEL_SIZE - offset_xy[0]),
                    snap_modulus(iy0 * VOXEL_SIZE - offset_xy[1]),
                    snap_modulus(iz0 * VOXEL_SIZE + z_min_phys),
                ],
                'box_max': [
                    snap_modulus(ix1 * VOXEL_SIZE - offset_xy[0]),
                    snap_modulus(iy1 * VOXEL_SIZE - offset_xy[1]),
                    snap_modulus(iz1 * VOXEL_SIZE + z_min_phys),
                ],
            })
    return boxes


print('Step 1b 就绪: 合成拓扑 + 条件生成评估函数 +', len(SYNTHETIC_PROBE_SPECS), '个探针')


In [ ]:
# Step 2: 批量预处理 JSON -> .pt（V4 画布 88×88×24 + 绝对数量条件 + QC）
os.makedirs(OUT_DIR, exist_ok=True)
meta_path = os.path.join(OUT_DIR, '_cache_meta.json')
qc_report_path = os.path.join(DATA_DIR, 'eval_reports', 'preprocess_qc_v4_tensors_88x88x24.json')

json_files = discover_json_files()
if not json_files:
    raise FileNotFoundError(f'未找到 JSON: {JSON_ROOT}（请确认 data/processed 已上传）')

need_rebuild = True
if os.path.exists(meta_path):
    with open(meta_path, 'r', encoding='utf-8') as f:
        meta = json.load(f)
    need_rebuild = meta.get('version') != TENSOR_CACHE_VERSION or meta.get('count') != len(json_files)

if need_rebuild:
    ok, fail, skip = 0, 0, 0
    qc_rows = []
    for fp in json_files:
        fname = os.path.basename(fp)
        try:
            with open(fp, 'r', encoding='utf-8') as f:
                data = json.load(f)
            qc = qc_sample_voxelization(data)
            if qc is None:
                skip += 1
                continue
            sample = json_to_sample(data)
            if sample is None:
                skip += 1
                continue
            sample['source_json'] = fp
            sample['pt_name'] = fname.replace('.json', '.pt')
            sample['qc'] = qc
            torch.save(sample, os.path.join(OUT_DIR, sample['pt_name']))
            qc_rows.append({'file': fname, **qc})
            ok += 1
        except Exception as exc:
            fail += 1
            print(f'[FAIL] {fname}: {exc}')
    clipped_any = sum(1 for r in qc_rows if r['clipped_rooms'] > 0)
    clipped_all = sum(1 for r in qc_rows if r['clipped_rooms'] == r['total_rooms'] and r['total_rooms'] > 0)
    qc_summary = {
        'version': TENSOR_CACHE_VERSION,
        'grid': '88x88x24',
        'count_ok': ok,
        'count_fail': fail,
        'count_skip': skip,
        'samples_with_clipped_rooms': clipped_any,
        'samples_fully_clipped': clipped_all,
        'clip_rate': clipped_any / max(ok, 1),
        'mean_occ_ratio': float(np.mean([r['occ_ratio'] for r in qc_rows])) if qc_rows else 0.0,
        'max_building_x': float(max((r['building_x'] for r in qc_rows), default=0)),
        'max_building_y': float(max((r['building_y'] for r in qc_rows), default=0)),
        'max_building_z': float(max((r['building_z'] for r in qc_rows), default=0)),
    }
    os.makedirs(os.path.dirname(qc_report_path), exist_ok=True)
    with open(qc_report_path, 'w', encoding='utf-8') as f:
        json.dump({'summary': qc_summary, 'per_sample': qc_rows}, f, indent=2, ensure_ascii=False)
    with open(meta_path, 'w', encoding='utf-8') as f:
        json.dump({
            'version': TENSOR_CACHE_VERSION,
            'count': len(json_files),
            'ok': ok,
            'fail': fail,
            'skip': skip,
            'json_root': JSON_ROOT,
            'grid': '88x88x24',
            'qc_report': qc_report_path,
        }, f, indent=2, ensure_ascii=False)
    print(f'预处理完成: 成功 {ok}, 失败 {fail}, 跳过 {skip}')
    print(f'QC: 有裁切房间样本 {clipped_any}/{ok} ({qc_summary["clip_rate"]*100:.2f}%) | 报告 {qc_report_path}')
else:
    print(f'命中缓存 {TENSOR_CACHE_VERSION}，跳过预处理（共 {len(json_files)} 个 JSON）')
    if os.path.exists(qc_report_path):
        with open(qc_report_path, 'r', encoding='utf-8') as f:
            qc_saved = json.load(f)
        print('QC 摘要:', qc_saved.get('summary', {}))


In [ ]:
# Step 3: V4 CVAE 模型（绝对数量条件 + CondMLP + FiLM + GroupNorm + 计数辅助头）
import math
LATENT_DIM = 256
HIDDEN_DIM = 128


def build_hetero_convs(in_dim, out_dim, vertical_gat=False):
    h_conv = {}
    node_types = ['living', 'service', 'circulation']
    for s in node_types:
        for d in node_types:
            h_conv[(s, 'horizontal', d)] = SAGEConv(in_dim, out_dim)
    vertical_pairs = [
        ('circulation', 'living'), ('living', 'circulation'),
        ('circulation', 'service'), ('service', 'circulation'),
        ('circulation', 'circulation'),
    ]
    for s, d in vertical_pairs:
        if vertical_gat:
            h_conv[(s, 'vertical', d)] = GATv2Conv(
                in_dim, out_dim, heads=2, concat=False, add_self_loops=False
            )
        else:
            h_conv[(s, 'vertical', d)] = SAGEConv(in_dim, out_dim)
    return HeteroConv(h_conv, aggr='sum')


def _valid_groups(groups, channels):
    g = min(groups, channels)
    while channels % g != 0:
        g -= 1
    return max(1, g)


class CondEncoder(nn.Module):
    """原始条件(绝对数量, COND_DIM) → cond_embed(COND_EMBED_DIM)。"""
    def __init__(self, cond_dim=COND_DIM, embed_dim=COND_EMBED_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_dim, embed_dim), nn.LayerNorm(embed_dim), nn.SiLU(),
            nn.Linear(embed_dim, embed_dim), nn.LayerNorm(embed_dim), nn.SiLU(),
        )

    def forward(self, cond):
        return self.net(cond)


class FiLM3d(nn.Module):
    """GroupNorm(affine=False) + 条件调制 (1+γ)·x + β；γ 初始≈0 → 初始恒等。"""
    def __init__(self, num_channels, embed_dim=COND_EMBED_DIM, groups=8):
        super().__init__()
        self.norm = nn.GroupNorm(_valid_groups(groups, num_channels), num_channels, affine=False)
        self.to_film = nn.Linear(embed_dim, num_channels * 2)
        nn.init.zeros_(self.to_film.weight)
        nn.init.zeros_(self.to_film.bias)

    def forward(self, x, cond_embed):
        x = self.norm(x)
        gamma, beta = self.to_film(cond_embed).chunk(2, dim=1)
        gamma = gamma.view(x.size(0), -1, 1, 1, 1)
        beta = beta.view(x.size(0), -1, 1, 1, 1)
        return x * (1.0 + gamma) + beta


class HeteroGraphVAEEncoder(nn.Module):
    """计数感知 pooling：mean-pool 拼接每超类节点计数，缓解 mean-pool 稀释。"""
    def __init__(self, node_in_dim=NODE_IN_DIM, hidden_dim=HIDDEN_DIM, latent_dim=LATENT_DIM):
        super().__init__()
        self.conv1 = build_hetero_convs(node_in_dim, hidden_dim, vertical_gat=True)
        self.conv2 = build_hetero_convs(hidden_dim, hidden_dim, vertical_gat=False)
        self.node_types = ['living', 'service', 'circulation']
        self.to_mu = nn.Linear(hidden_dim + len(self.node_types), latent_dim)
        self.to_logvar = nn.Linear(hidden_dim + len(self.node_types), latent_dim)

    def _device(self, x_dict):
        for x in x_dict.values():
            return x.device
        return torch.device('cpu')

    def _pool(self, x_dict, batch_dict):
        bs = 1
        for b in batch_dict.values():
            if b.numel() > 0:
                bs = max(bs, int(b.max()) + 1)
        dev = self._device(x_dict)
        feats, counts = [], []
        for nt in self.node_types:
            if nt in batch_dict and nt in x_dict and x_dict[nt].size(0) > 0:
                feats.append(global_mean_pool(x_dict[nt], batch_dict[nt], size=bs))
                c = torch.bincount(batch_dict[nt], minlength=bs).float().unsqueeze(1)
            else:
                feats.append(torch.zeros(bs, HIDDEN_DIM, device=dev))
                c = torch.zeros(bs, 1, device=dev)
            counts.append(c)
        h = torch.stack(feats, dim=0).mean(dim=0)            # (bs, hidden)
        cnt = torch.cat(counts, dim=1) / COND_COUNT_SCALE     # (bs, 3)
        return torch.cat([h, cnt], dim=1)

    def forward(self, x_dict, edge_index_dict, batch_dict):
        x_dict = {k: torch.relu(self.conv1(x_dict, edge_index_dict)[k]) for k in x_dict}
        x_dict = {k: torch.relu(self.conv2(x_dict, edge_index_dict)[k]) for k in x_dict}
        h = self._pool(x_dict, batch_dict)
        return self.to_mu(h), self.to_logvar(h)


class ConditionalVoxelDecoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, cond_dim=COND_DIM, channels=NUM_CHANNELS):
        super().__init__()
        self.init_volume_size = (256, DECODER_INIT_X, DECODER_INIT_Y, DECODER_INIT_Z)
        self.cond_mlp = CondEncoder(cond_dim, COND_EMBED_DIM)
        self.fc = nn.Linear(latent_dim + COND_EMBED_DIM, int(np.prod(self.init_volume_size)))
        self.deconv1 = nn.ConvTranspose3d(256, 128, 4, stride=2, padding=1)
        self.film1 = FiLM3d(128)
        self.deconv2 = nn.ConvTranspose3d(128, 64, 4, stride=2, padding=1)
        self.film2 = FiLM3d(64)
        self.deconv3 = nn.ConvTranspose3d(64, 32, 4, stride=2, padding=1)
        self.film3 = FiLM3d(32)
        self.deconv4 = nn.ConvTranspose3d(32, channels, 4, stride=2, padding=1)

    def forward(self, z, cond):
        ce = self.cond_mlp(cond)
        d = self.fc(torch.cat([z, ce], dim=-1)).view(z.size(0), *self.init_volume_size)
        h = torch.relu(self.film1(self.deconv1(d), ce))
        h = torch.relu(self.film2(self.deconv2(h), ce))
        h = torch.relu(self.film3(self.deconv3(h), ce))
        x = self.deconv4(h)
        return x[:, :, :RES_X, :RES_Y, :RES_Z]


class SpatialModalCVAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = HeteroGraphVAEEncoder()
        self.decoder = ConditionalVoxelDecoder()
        self.count_head = nn.Sequential(
            nn.Linear(LATENT_DIM, 128), nn.SiLU(), nn.Linear(128, NUM_ROOM_TYPES)
        )

    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu

    def predict_counts(self, z):
        return self.count_head(z)

    def forward(self, batch):
        mu, logvar = self.encoder(
            batch.x_dict, batch.edge_index_dict, graph_batch_dict(batch)
        )
        z = self.reparameterize(mu, logvar)
        cond = graph_condition(batch)
        logits = self.decoder(z, cond)
        return logits, mu, logvar


model = SpatialModalCVAE().to(device)
print(f'V4 参数量: {sum(p.numel() for p in model.parameters()):,}')


In [ ]:
# Step 4: 训练（AMP + 梯度积累 + KL Annealing + 拓扑加噪 + 合成探针）
import gc
from torch.utils.data import Dataset

_need = ['set_seed', 'model', 'forward_model', 'SPLIT_SEED', 'OUT_DIR', 'augment_batch_topology', 'run_synthetic_probe_eval']
_missing = [k for k in _need if k not in globals()]
if _missing:
    raise RuntimeError(
        '请先运行 Step 1 → Step 1b → Step 3，再执行 Step 4。\n'
        '（内核崩溃/重启后内存会清空，不能直接单跑本 cell）\n'
        '缺少: ' + ', '.join(_missing)
    )

set_seed(SPLIT_SEED)

class LazyPtGraphDataset(Dataset):
    """按 batch 从磁盘懒加载 .pt，避免 468 样本全进 RAM（约 6GB+）。"""
    def __init__(self, entries):
        self.entries = entries  # [(pt_name, filepath), ...]

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        pt_name, fp = self.entries[idx]
        sample = torch.load(fp, map_location='cpu', weights_only=False)
        hg = sample['graph']
        hg.y = sample['voxel']
        cond = sample['condition']
        hg.condition = cond.unsqueeze(0) if cond.dim() == 1 else cond
        hg.pt_name = sample.get('pt_name') or pt_name
        return hg


pt_files = sorted([
    p for p in glob.glob(os.path.join(OUT_DIR, '*.pt'))
    if not os.path.basename(p).startswith('_')
])
if not pt_files:
    raise RuntimeError('请先运行 Step 2 预处理')

items = [(os.path.basename(fp), fp) for fp in pt_files]
pt_names = [x[0] for x in items]
train_names, val_names, split_meta = load_train_val_split(pt_names)
train_name_set, val_name_set = set(train_names), set(val_names)
train_entries = [x for x in items if x[0] in train_name_set]
val_entries = [x for x in items if x[0] in val_name_set]
print(f'样本总数 {len(items)} | 训练 {len(train_entries)} | 验证 {len(val_entries)}')
print(f'划分文件: {split_path()}')
print('数据加载: 懒加载（按 batch 读盘，节省 RAM）')

train_loader = GraphDataLoader(LazyPtGraphDataset(train_entries), batch_size=BATCH_SIZE, shuffle=True)
val_loader = GraphDataLoader(LazyPtGraphDataset(val_entries), batch_size=BATCH_SIZE, shuffle=False) if val_entries else None

optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=8)
_use_amp = USE_AMP and device.type == 'cuda'
scaler = torch.amp.GradScaler('cuda', enabled=_use_amp)

# v2 新增：LR Warmup（前5个epoch学习率从1e-5线性升到LR）
WARMUP_EPOCHS = 5
WARMUP_LR_START = 1e-5

def set_lr_for_epoch(epoch):
    """前 WARMUP_EPOCHS epoch 线性 warmup，之后交 scheduler 管理"""
    if epoch <= WARMUP_EPOCHS:
        lr = WARMUP_LR_START + (LR - WARMUP_LR_START) * (epoch / WARMUP_EPOCHS)
        for g in optimizer.param_groups:
            g['lr'] = lr
        return lr
    return optimizer.param_groups[0]['lr']

import time as _time
save_path = weight_path()
ckpt_path = checkpoint_path()

best_val = float('inf')
bad_epochs = 0
history = {'train': [], 'val': [], 'kl_beta': [], 'synth_probe': []}
best_probe_score = -1.0
probe_save_path = probe_weight_path()
_train_start = _time.time()
start_epoch = 1


def save_training_checkpoint(epoch, tag='latest'):
    payload = {
        'epoch': epoch,
        'cache_version': CACHE_VERSION,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'scaler': scaler.state_dict(),
        'best_val': best_val,
        'bad_epochs': bad_epochs,
        'history': history,
        'train_start': _train_start,
        'train_count': len(train_entries),
        'val_count': len(val_entries),
        'tag': tag,
    }
    tmp_path = ckpt_path + '.tmp'
    torch.save(payload, tmp_path)
    if os.path.exists(ckpt_path):
        os.remove(ckpt_path)
    os.rename(tmp_path, ckpt_path)


def try_resume_training():
    global start_epoch, best_val, bad_epochs, history, _train_start
    if FORCE_NEW_TRAINING:
        print('FORCE_NEW_TRAINING=True，从头训练')
        return
    if not RESUME_IF_CHECKPOINT:
        print('RESUME_IF_CHECKPOINT=False，从头训练')
        return
    if not os.path.exists(ckpt_path):
        if MANUAL_RESUME_EPOCH > 0 and os.path.exists(save_path):
            model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
            start_epoch = int(MANUAL_RESUME_EPOCH) + 1
            print(f'⚠ 无检查点，从最佳权重续训: 已完成 epoch {MANUAL_RESUME_EPOCH} → 从 {start_epoch} 继续')
            print('  （优化器状态未恢复，loss 可能短暂抖动；之后每 epoch 会自动存检查点）')
            return
        print('无检查点，从头训练')
        return
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    if ckpt.get('cache_version') != CACHE_VERSION:
        print(f"检查点版本 {ckpt.get('cache_version')} != {CACHE_VERSION}，从头训练")
        return
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    scaler.load_state_dict(ckpt['scaler'])
    start_epoch = int(ckpt['epoch']) + 1
    best_val = float(ckpt['best_val'])
    bad_epochs = int(ckpt['bad_epochs'])
    history = ckpt.get('history', history)
    _train_start = ckpt.get('train_start', _time.time())
    torch.save(model.state_dict(), save_path)
    print(f"✓ 断点续训: 已完成 epoch {ckpt['epoch']} → 从 {start_epoch}/{EPOCHS} 继续")
    print(f"  best_val={best_val:.4f} | bad_epochs={bad_epochs} | 检查点: {ckpt_path}")


try_resume_training()


def kl_beta_for_epoch(epoch):
    if KL_ANNEAL_EPOCHS <= 0:
        return KL_WEIGHT_MAX
    t = min(1.0, epoch / KL_ANNEAL_EPOCHS)
    return KL_WEIGHT_MAX * t


def compute_batch_loss(batch, kl_weight):
    bs = graph_batch_size(batch)
    target = batch.y.view(bs, NUM_CHANNELS, RES_X, RES_Y, RES_Z)
    logits, mu, logvar = forward_model(model, batch)
    bce = F.binary_cross_entropy_with_logits(logits, target)
    dice = dice_loss_with_logits(logits, target)
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    # V4: 计数辅助损失（mu → 房间数），监督潜变量编码房间程序
    cond = graph_condition(batch)
    target_counts = cond[:, 3:3 + NUM_ROOM_TYPES] * COND_COUNT_SCALE
    aux = F.smooth_l1_loss(model.predict_counts(mu), target_counts)
    loss = bce + DICE_WEIGHT * dice + kl_weight * kl + AUX_COUNT_WEIGHT * aux
    return loss, bs


def run_epoch(loader, train_mode=True, epoch=1):
    model.train(train_mode)
    total = 0.0
    n = 0
    kl_weight = kl_beta_for_epoch(epoch) if train_mode else KL_WEIGHT_MAX
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(loader, start=1):
        batch = batch.to(device)
        if train_mode and AUGMENT_PROB > 0 and random.random() < AUGMENT_PROB:
            batch = augment_batch_topology(batch)

        with torch.amp.autocast('cuda', enabled=_use_amp):
            loss, bs = compute_batch_loss(batch, kl_weight)
            loss_scaled = loss / ACCUM_STEPS

        if train_mode:
            scaler.scale(loss_scaled).backward()
            if step % ACCUM_STEPS == 0 or step == len(loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
        total += loss.item() * bs
        n += bs
        del batch, loss, loss_scaled
    if train_mode and device.type == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()
    return total / max(n, 1), kl_weight


if start_epoch > EPOCHS:
    print(f'检查点显示已完成 {EPOCHS} epoch，跳过训练。请运行 Step 5 查看摘要。')
    epoch = EPOCHS
else:
    print(f'开始训练 ({device}) | V4 {RES_X}×{RES_Y}×{RES_Z} | AMP={USE_AMP and device.type == "cuda"} | accum={ACCUM_STEPS} | eff_batch≈{BATCH_SIZE * ACCUM_STEPS}')
    print(f'epoch 范围: {start_epoch}–{EPOCHS} | 检查点: {ckpt_path}')
    for epoch in range(start_epoch, EPOCHS + 1):
        lr_now = set_lr_for_epoch(epoch)
        tr, beta = run_epoch(train_loader, True, epoch)
        history['train'].append(tr)
        history['kl_beta'].append(beta)
        va, _ = run_epoch(val_loader, False, epoch) if val_loader else (tr, beta)
        history['val'].append(va)
        if epoch > WARMUP_EPOCHS:
            scheduler.step(va)

        if va < best_val:
            best_val = va
            bad_epochs = 0
            torch.save(model.state_dict(), save_path)
        else:
            bad_epochs += 1

        save_training_checkpoint(epoch)

        show_log = (epoch == start_epoch or epoch % LOG_EVERY == 0 or epoch % 10 == 0)
        if show_log:
            print(f'Epoch {epoch:03d}/{EPOCHS} | lr={lr_now:.2e} | train={tr:.4f} val={va:.4f} best={best_val:.4f} beta={beta:.2e} | ckpt✓')
        if epoch % PROBE_EVERY == 0:
            print('  [合成拓扑探针]')
            probe = run_synthetic_probe_eval(model, verbose=True)
            history['synth_probe'].append({'epoch': epoch, **probe})
            if float(probe.get('mean_n_occ', 0)) > best_probe_score:
                best_probe_score = float(probe['mean_n_occ'])
                torch.save(model.state_dict(), probe_save_path)
                print(f'  [探针最佳] mean_n_occ={best_probe_score:.0f} → {probe_save_path}')

        if bad_epochs >= PATIENCE:
            print(f'早停于 epoch {epoch}，最佳 val={best_val:.4f}')
            break

    done_path = ckpt_path.replace('.pt', '_done.pt')
    if os.path.exists(ckpt_path):
        os.rename(ckpt_path, done_path)
        print(f'训练结束，检查点归档: {done_path}')

set_experiment_meta(
    train_count=len(train_entries),
    total_count=len(items),
    extra={
        'model_version': 'v4',
        'grid': f'{RES_X}x{RES_Y}x{RES_Z}',
        'init_volume': [256, DECODER_INIT_X, DECODER_INIT_Y, DECODER_INIT_Z],
        'cond_semantics': 'absolute_counts',
        'cond_count_scale': COND_COUNT_SCALE,
        'cond_embed_dim': COND_EMBED_DIM,
        'decoder_norm': 'groupnorm+film',
        'aux_count_weight': AUX_COUNT_WEIGHT,
        'best_probe_mean_n_occ': float(best_probe_score),
        'weight_probe_file': WEIGHT_PROBE_FILENAME,
        'val_samples': len(val_entries),
        'epochs_run': epoch,
        'best_val_loss': float(best_val),
        'train_seconds': round(_time.time() - _train_start, 1),
        'cache_version': CACHE_VERSION,
        'weight_file': WEIGHT_FILENAME,
        'split_file': SPLIT_FILENAME,
        'batch_size': BATCH_SIZE,
        'accum_steps': ACCUM_STEPS,
        'use_amp': USE_AMP,
        'kl_weight_max': KL_WEIGHT_MAX,
        'kl_anneal_epochs': KL_ANNEAL_EPOCHS,
        'edge_drop_prob': EDGE_DROP_PROB,
        'augment_prob': AUGMENT_PROB,
        'synth_probe': history['synth_probe'],
        'train_names': train_names,
        'val_names': val_names,
    },
)
meta_path = experiment_report_path('experiment_meta', ext='json')
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(EXPERIMENT_META, f, indent=2, ensure_ascii=False)
print(f'最佳权重已保存: {save_path}')
print(f'实验元数据: {meta_path}')
print(EXPERIMENT_META)

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train'], label='train')
axes[0].plot(history['val'], label='val')
axes[0].set_title(f'V4 Loss ({RES_X}×{RES_Y}×{RES_Z})')
axes[0].set_xlabel('Epoch')
axes[0].grid(True, alpha=0.3)
axes[0].legend()
axes[1].plot(history['kl_beta'], label='kl_beta', color='tab:orange')
axes[1].set_title('KL beta schedule')
axes[1].set_xlabel('Epoch')
axes[1].grid(True, alpha=0.3)
axes[1].legend()
plt.tight_layout()

curve_path = experiment_report_path('training_curve', ext='png')
fig.savefig(curve_path, dpi=120, bbox_inches='tight')
print(f'训练曲线已保存: {curve_path}')

append_training_run_log({
    'trained_at': EXPERIMENT_META.get('trained_at'),
    'model_version': 'v4',
    'grid': f'{RES_X}x{RES_Y}x{RES_Z}',
    'cache_version': CACHE_VERSION,
    'epochs_run': epoch,
    'best_val_loss': float(best_val),
    'best_probe_mean_n_occ': float(best_probe_score),
    'train_seconds': EXPERIMENT_META.get('train_seconds'),
    'weight_file': WEIGHT_FILENAME,
    'weight_probe_file': WEIGHT_PROBE_FILENAME,
    'meta_file': meta_path,
    'curve_file': curve_path,
})
print(f'训练记录已追加: {RUNS_LOG_PATH}')

plt.show()


## Step D: V4-a 诊断（后验坍缩 / 语义敏感性 / 解码器是否听条件）

在**已有 V3 权重**时运行即可，**不必重新训练**。最少只需 Step 1 → 1b → 3 → **Step D**（会从磁盘加载 `spatial_modal_cvae_v3_88x88x24.pth`）。若同会话未跑 Step 4，A 组会从 `OUT_DIR` 的 `.pt` 回退统计。

目的是在投入 V4 大改之前，**先量出真实失败模式**，避免误把"条件被忽略"当成"后验坍缩"来修。

产出三组指标，写入 `data/eval_reports/diagnostics_*.json`：

| 组 | 指标 | 判读 |
|---|---|---|
| A 坍缩 | `mu_norm` / `sigma_mean` / `kl_per_dim` / `dead_dims_ratio` / `active_units_ratio` | `sigma≈1 且 kl≈0 且 dead≈1` → 经典坍缩；`sigma 偏小 + kl 大` → AE 化（z 携带过多） |
| B 编码器语义敏感性 | 改卧室数(3↔5)/改场地后 μ 的余弦距离 | 距离过小（<0.02）→ 编码器对条件变化不敏感 |
| C 解码器条件敏感性 | 固定 z 只改 cond 的体素变化率 vs 改 z 的变化率 vs 抹零 cond | **若改 cond 几乎不变、改 z 大变 → 解码器无视条件（真正病灶）** |

> 诊断**不修改权重**，可反复运行。建议先加载"最佳 val 权重"再诊断（默认用当前内存模型）。

In [ ]:
# Step D: V4-a 诊断（坍缩 / 语义 / 条件敏感性）
import torch
import torch.nn.functional as F
import numpy as np
import json, os

_need_d = ['model', 'device', 'graph_batch_dict', 'prepare_graph_batch', 'json_to_sample',
           'get_node_supertype', 'build_user_request', 'layout_rooms_from_program',
           'request_to_house_json', 'LATENT_DIM', 'NUM_CHANNELS']
_miss_d = [k for k in _need_d if k not in globals()]
if _miss_d:
    raise RuntimeError('请先运行 Step 1 → 1b → 3（不必重训 Step 4）。缺少: ' + ', '.join(_miss_d))

# 优先用最佳 val 权重诊断（更代表早停所选模型）；失败则用当前内存模型
try:
    _bp = weight_path()
    if os.path.exists(_bp):
        model.load_state_dict(torch.load(_bp, map_location=device, weights_only=True))
        print(f'[诊断] 已加载最佳 val 权重: {_bp}')
    else:
        print('[诊断] 未找到最佳权重文件，使用当前内存模型')
except Exception as _e:
    print(f'[诊断] 加载最佳权重失败({_e})，使用当前内存模型')
model.eval()

DEAD_KL_THRESH = 0.01      # 单维平均 KL 低于此值视为"死维"
ACTIVE_VAR_THRESH = 0.01   # mu 在样本间方差高于此值视为"活跃单元"(Burda et al.)


@torch.no_grad()
def _encode_mu_logvar(graph, cond):
    batch = prepare_graph_batch(graph, condition=cond).to(device)
    mu, logvar = model.encoder(batch.x_dict, batch.edge_index_dict, graph_batch_dict(batch))
    return mu.detach(), logvar.detach()


@torch.no_grad()
def _decode_cls(z, cond):
    c = cond.unsqueeze(0) if cond.dim() == 1 else cond
    logits = model.decoder(z.to(device), c.to(device))
    return torch.argmax(logits[0], dim=0).cpu().numpy()


def _change_fraction(a, b):
    return float(np.mean(a != b))


def _occ_by_type(cls):
    return {int(c): int((cls == c).sum()) for c in range(1, NUM_CHANNELS) if (cls == c).any()}


def _append_hetero_edge(edges_dict, src_t, rel, dst_t, si, di):
    edges_dict.setdefault((src_t, rel, dst_t), []).append([si, di])


def _merge_program_topology_edges(rooms, program_graph, program_edge_types, id_to_idx, edges_dict):
    if program_graph is None:
        return
    for u, v in program_graph.edges:
        if u not in id_to_idx or v not in id_to_idx:
            continue
        t1, idx1 = id_to_idx[u]
        t2, idx2 = id_to_idx[v]
        rel = (program_edge_types or {}).get((u, v), 'horizontal')
        if rel not in ('horizontal', 'vertical'):
            rel = 'horizontal'
        for src_t, dst_t, si, di in ((t1, t2, idx1, idx2), (t2, t1, idx2, idx1)):
            _append_hetero_edge(edges_dict, src_t, rel, dst_t, si, di)


def _sample_with_program_topo(data, program_graph, program_edge_types):
    """合成布局常无几何邻接边，需注入程序拓扑（与 spatial_modal_infer 一致）。"""
    sample = json_to_sample(data)
    if sample is None:
        return None
    hg = sample['graph']
    if len(hg.edge_types) > 0:
        return sample
    rooms = data.get('rooms', [])
    id_to_idx = {}
    counters = {'living': 0, 'service': 0, 'circulation': 0}
    for r in rooms:
        ntype = get_node_supertype(r['type'])
        id_to_idx[r['id']] = (ntype, counters[ntype])
        counters[ntype] += 1
    edges_dict = {}
    _merge_program_topology_edges(rooms, program_graph, program_edge_types, id_to_idx, edges_dict)
    if not edges_dict:
        raise RuntimeError('图无任何边：请检查 layout / 程序拓扑')
    for key, elist in edges_dict.items():
        hg[key].edge_index = torch.tensor(elist, dtype=torch.long).t().contiguous()
    return sample


# ---------- A. 坍缩指标（val 集或磁盘 .pt 回退）----------
print('\n========== A. 后验坍缩诊断 ==========')
mus, logvars = [], []
_collapse_source = None
if 'val_loader' in globals() and val_loader is not None:
    _collapse_source = 'val_loader'
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            mu, logvar = model.encoder(batch.x_dict, batch.edge_index_dict, graph_batch_dict(batch))
            mus.append(mu.cpu()); logvars.append(logvar.cpu())
elif 'val_entries' in globals() and val_entries:
    _collapse_source = f'val_pt({len(val_entries)})'
    with torch.no_grad():
        for _, fp in val_entries:
            s = torch.load(fp, map_location='cpu', weights_only=False)
            mu, logvar = _encode_mu_logvar(s['graph'], s['condition'])
            mus.append(mu.cpu()); logvars.append(logvar.cpu())
elif 'OUT_DIR' in globals() and os.path.isdir(OUT_DIR):
    _pt_files = sorted(
        p for p in glob.glob(os.path.join(OUT_DIR, '*.pt'))
        if not os.path.basename(p).startswith('_')
    )
    if _pt_files:
        _collapse_source = f'pt_all({min(len(_pt_files), 40)})'
        with torch.no_grad():
            for fp in _pt_files[:40]:
                s = torch.load(fp, map_location='cpu', weights_only=False)
                mu, logvar = _encode_mu_logvar(s['graph'], s['condition'])
                mus.append(mu.cpu()); logvars.append(logvar.cpu())

if mus:
    mu_all = torch.cat(mus); logvar_all = torch.cat(logvars)
    sigma_all = torch.exp(0.5 * logvar_all)
    kl_dim = 0.5 * (mu_all.pow(2) + sigma_all.pow(2) - 1.0 - logvar_all)  # (N, D)
    kl_per_dim = kl_dim.mean(0)                                          # (D,)
    mu_norm = mu_all.norm(dim=1)
    dead_ratio = float((kl_per_dim < DEAD_KL_THRESH).float().mean())
    active_ratio = float((mu_all.var(0) > ACTIVE_VAR_THRESH).float().mean())
    collapse = {
        'source': _collapse_source,
        'n_samples': int(mu_all.size(0)),
        'latent_dim': int(mu_all.size(1)),
        'mu_norm_mean': float(mu_norm.mean()),
        'mu_norm_std': float(mu_norm.std()),
        'sigma_mean': float(sigma_all.mean()),
        'sigma_std': float(sigma_all.std()),
        'kl_total_mean': float(kl_per_dim.sum()),
        'kl_per_dim_max': float(kl_per_dim.max()),
        'kl_per_dim_top10': [round(v, 4) for v in torch.sort(kl_per_dim, descending=True).values[:10].tolist()],
        'dead_dims_ratio': dead_ratio,
        'active_units_ratio': active_ratio,
        'kl_per_dim': [round(v, 6) for v in kl_per_dim.tolist()],
    }
    print(f"来源={_collapse_source} | 样本 {collapse['n_samples']} | 潜维 {collapse['latent_dim']}")
    print(f"mu_norm = {collapse['mu_norm_mean']:.3f} ± {collapse['mu_norm_std']:.3f}")
    print(f"sigma_mean = {collapse['sigma_mean']:.4f}  | KL_total = {collapse['kl_total_mean']:.3f}")
    print(f"死维占比(KL<{DEAD_KL_THRESH}) = {dead_ratio*100:.1f}%  | 活跃单元占比 = {active_ratio*100:.1f}%")
    print(f"KL_per_dim Top10 = {collapse['kl_per_dim_top10']}")
else:
    collapse = {'error': '无 val_loader / val_entries / OUT_DIR .pt，请先跑 Step 2'}
    print('跳过 A：无可用验证样本（跑 Step 2 预处理后可从磁盘 .pt 回退）')


# ---------- B. 编码器语义敏感性 ----------
print('\n========== B. 编码器语义敏感性（改条件后 μ 的余弦距离）==========')
_BASE = {'entryway': 1, 'living_room': 1, 'dining_room': 1, 'kitchen': 1,
         'bedroom': 3, 'bathroom': 2, 'corridor': 2, 'stairs': 1, 'balcony': 1}
_SEED = 22


_ENC_CACHE = {}


def _encode_for(site_x, site_y, counts, seed=_SEED):
    """返回 device 上的 mu/logvar(含 batch 维) 与 cond。"""
    key = (site_x, site_y, tuple(sorted(counts.items())), seed)
    if key not in _ENC_CACHE:
        req = build_user_request(site_x, site_y, counts)
        rooms, G, pos, et = layout_rooms_from_program(req, seed=seed)
        data = request_to_house_json(req, rooms)
        sample = _sample_with_program_topo(data, G, et)
        mu, logvar = _encode_mu_logvar(sample['graph'], sample['condition'])
        _ENC_CACHE[key] = (mu.detach(), logvar.detach(), sample['condition'])
    return _ENC_CACHE[key]


def _mu_for(site_x, site_y, counts, seed=_SEED):
    mu, logvar, cond = _encode_for(site_x, site_y, counts, seed)
    return mu.squeeze(0).cpu(), cond


def _cos_dist(a, b):
    return float(1.0 - F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item())


mu_b3, cond_b3 = _mu_for(18000, 15000, {**_BASE, 'bedroom': 3})
mu_b5, _       = _mu_for(18000, 15000, {**_BASE, 'bedroom': 5})
mu_small, _    = _mu_for(12000, 9000,  _BASE)
mu_large, _    = _mu_for(28000, 22000, _BASE)
semantic = {
    'bedroom_3_vs_5_cos_dist': _cos_dist(mu_b3, mu_b5),
    'bedroom_3_vs_5_l2': float((mu_b3 - mu_b5).norm()),
    'site_small_vs_large_cos_dist': _cos_dist(mu_small, mu_large),
}
print(f"卧室 3↔5  μ 余弦距离 = {semantic['bedroom_3_vs_5_cos_dist']:.4f} (L2={semantic['bedroom_3_vs_5_l2']:.3f})")
print(f"场地 小↔大 μ 余弦距离 = {semantic['site_small_vs_large_cos_dist']:.4f}")


# ---------- C. 解码器条件敏感性（用真实 μ 做基准 z，并报 n_occ）----------
print('\n========== C. 解码器是否真听条件 ==========')
mu3_dev, logvar3_dev, cond_b3 = _encode_for(18000, 15000, {**_BASE, 'bedroom': 3})
mu5_dev, logvar5_dev, cond_b5c = _encode_for(18000, 15000, {**_BASE, 'bedroom': 5})


def _decode_occ(z, cond):
    cls = _decode_cls(z, cond)
    return cls, int((cls > 0).sum())


# 基准：真实生成场景用的就是 encoder 的 μ（mu_norm≈7.6，在训练流形上）
cls_mu3_c3, occ_mu3_c3 = _decode_occ(mu3_dev, cond_b3)           # 完整真实生成 (z=μ3, cond3)
cls_mu3_c5, occ_mu3_c5 = _decode_occ(mu3_dev, cond_b5c)          # 固定 z=μ3，只改 cond→5
cls_mu5_c3, occ_mu5_c3 = _decode_occ(mu5_dev, cond_b3)           # 固定 cond3，只改 z→μ5
cls_mu5_c5, occ_mu5_c5 = _decode_occ(mu5_dev, cond_b5c)          # 完整真实生成 (z=μ5, cond5)
cls_mu3_c0, occ_mu3_c0 = _decode_occ(mu3_dev, torch.zeros_like(cond_b3))  # 抹零 cond
# 离流形对照：z=0 与 z=randn（解释上一轮的全 0）
torch.manual_seed(0)
cls_z0_c3, occ_z0 = _decode_occ(torch.zeros(1, LATENT_DIM, device=device), cond_b3)
cls_zr_c3, occ_zr = _decode_occ(torch.randn(1, LATENT_DIM, device=device), cond_b3)

frac_cond = _change_fraction(cls_mu3_c3, cls_mu3_c5)    # 固定 z=μ3，只改 cond
frac_z = _change_fraction(cls_mu3_c3, cls_mu5_c3)       # 固定 cond3，只改 z
frac_condzero = _change_fraction(cls_mu3_c3, cls_mu3_c0)
ratio = frac_cond / max(frac_z, 1e-6)
decoder = {
    'occ_mu3_cond3': occ_mu3_c3, 'occ_mu3_cond5': occ_mu3_c5,
    'occ_mu5_cond3': occ_mu5_c3, 'occ_mu5_cond5': occ_mu5_c5,
    'occ_mu3_cond_zeroed': occ_mu3_c0,
    'occ_z0_cond3': occ_z0, 'occ_zrandn_cond3': occ_zr,
    'voxel_change_frac_cond(bed3->5)@mu': round(frac_cond, 5),
    'voxel_change_frac_z(mu3->mu5)': round(frac_z, 5),
    'voxel_change_frac_cond_zeroed@mu': round(frac_condzero, 5),
    'cond_over_z_ratio': round(ratio, 4),
    'occ_by_type_mu3_cond3': _occ_by_type(cls_mu3_c3),
    'occ_by_type_mu5_cond5': _occ_by_type(cls_mu5_c5),
}
print(f"非空体素 n_occ:  μ3/c3={occ_mu3_c3}  μ3/c5={occ_mu3_c5}  μ5/c3={occ_mu5_c3}  μ5/c5={occ_mu5_c5}")
print(f"             抹零cond={occ_mu3_c0}  | 离流形 z0={occ_z0}  z~N={occ_zr}")
print(f"改 cond(卧室3→5)@μ 体素变化率 = {frac_cond*100:.3f}%")
print(f"改 z(μ3→μ5)        体素变化率 = {frac_z*100:.3f}%")
print(f"抹零 cond@μ        体素变化率 = {frac_condzero*100:.3f}%")
print(f"cond / z 影响比    = {ratio:.3f}  (越小说明解码器越无视条件)")
if max(occ_mu3_c3, occ_mu5_c5) == 0:
    print('⚠ 真实 μ 下 n_occ=0：模型 argmax 全 empty → V4 损失需对抗稀疏塌空(类别加权/Focal/提高Dice)')


# ---------- 判读与建议 ----------
verdict = []
if isinstance(collapse, dict) and 'dead_dims_ratio' in collapse:
    if collapse['dead_dims_ratio'] > 0.9 and collapse['kl_total_mean'] < 1.0:
        verdict.append('CONFIRMED_POSTERIOR_COLLAPSE: z≈先验，应上 KL退火/FreeBits（V4 P0 成立）')
    elif collapse['sigma_mean'] < 0.3 and collapse['kl_total_mean'] > 20:
        verdict.append('AE_LIKE: z 携带过多信息（β=5e-3 太小），FreeBits 可能加剧条件被忽略')
if semantic['bedroom_3_vs_5_cos_dist'] < 0.02:
    verdict.append('ENCODER_INSENSITIVE: 编码器 μ 对卧室数变化几乎不响应（疑似 mean-pool 稀释）')
_real_occ = max(decoder['occ_mu3_cond3'], decoder['occ_mu5_cond5'])
if _real_occ == 0:
    verdict.append('EMPTY_COLLAPSE: 真实 μ 下 argmax 全 empty → 主因是稀疏占用塌空，V4 损失须对抗（类别加权/Focal/提高Dice），并复核占用率')
elif ratio < 0.2 or frac_cond < 0.01:
    verdict.append('DECODER_IGNORES_CONDITION: 解码器主要靠 z，条件被淹没 → 优先 V4-b（绝对数量+FiLM+升维）')
if not verdict:
    verdict.append('未触发明显告警阈值，请人工对照 A/B/C 数值判断')
print('\n========== 判读 ==========')
for v in verdict:
    print('  •', v)

report = {
    'phase': 'V4-a_diagnostics',
    'weight_file': WEIGHT_FILENAME,
    'trained_at': EXPERIMENT_META.get('trained_at') if 'EXPERIMENT_META' in globals() else None,
    'thresholds': {'dead_kl': DEAD_KL_THRESH, 'active_var': ACTIVE_VAR_THRESH},
    'A_collapse': collapse,
    'B_semantic': semantic,
    'C_decoder': decoder,
    'verdict': verdict,
}
_diag_path = experiment_report_path('diagnostics', ext='json')
with open(_diag_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print(f'\n诊断报告已保存: {_diag_path}')

## Step 5: 训练结果摘要

本 notebook **不包含 3D 可视化**。请下载权重到本地，使用 `scripts/spatial_modal_infer/`：

```bash
cd scripts/spatial_modal_infer
python run_cli.py --weights /path/to/spatial_modal_cvae_v4_88x88x24.pth \
  --site-x 18000 --site-y 15000 --seed 123 \
  --living-room 1 --bedroom 3 --kitchen 1 --out-dir ./outputs
```

实验记录目录：`data/eval_reports/`（`experiment_meta_*.json`、`training_runs.jsonl`、训练曲线 PNG）。


In [ ]:
# Step 5: 训练结果摘要与实验索引
import glob
import json
import os

print('=' * 60)
print('V3 训练产物')
print('=' * 60)
for label, path_fn in [
    ('最佳 val 权重', weight_path),
    ('最佳探针权重', probe_weight_path),
    ('断点归档', lambda: checkpoint_path().replace('.pt', '_done.pt')),
]:
    p = path_fn() if callable(path_fn) else path_fn
    status = '✓' if os.path.exists(p) else '✗'
    print(f'  [{status}] {label}: {p}')

meta_glob = sorted(glob.glob(os.path.join(DATA_DIR, 'eval_reports', 'experiment_meta_*.json')))
if meta_glob:
    with open(meta_glob[-1], 'r', encoding='utf-8') as f:
        meta = json.load(f)
    print('\n最近一次实验 meta:')
    for k in ['model_version', 'grid', 'epochs_run', 'best_val_loss', 'best_probe_mean_n_occ',
              'train_seconds', 'weight_file', 'trained_at']:
        if k in meta:
            print(f'  {k}: {meta[k]}')
    curve = meta_glob[-1].replace('experiment_meta_', 'training_curve_').replace('.json', '.png')
    if os.path.exists(curve):
        print(f'  训练曲线: {curve}')

if os.path.exists(RUNS_LOG_PATH):
    print(f'\n训练历史索引: {RUNS_LOG_PATH}')
    with open(RUNS_LOG_PATH, 'r', encoding='utf-8') as f:
        lines = [ln for ln in f if ln.strip()]
    print(f'  共 {len(lines)} 次训练记录（最近 3 条）:')
    for ln in lines[-3:]:
        rec = json.loads(ln)
        print(f"    {rec.get('trained_at')} | ep={rec.get('epochs_run')} | best_val={rec.get('best_val_loss')} | grid={rec.get('grid')}")
else:
    print('\n尚无 training_runs.jsonl，完成 Step 4 后会自动追加。')
